# Mean GMAL score per Local Authority District

Joins each GMAL grid point (`gmal_2016.json`, `gmal_2026.json`) to its containing GM borough polygon (`gm_lad.geojson`) and computes the mean of every score field per LAD per year. Output is written to `gmal_lad_means.csv` in this folder.

In [1]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

DATA_DIR = Path('.').resolve()
print('Working in:', DATA_DIR)

Working in: C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\accessibility


In [2]:
# Load LAD polygons
lads = gpd.read_file(DATA_DIR / 'gm_lad.geojson')
lads = lads[['LADNM', 'geometry']].set_crs(4326, allow_override=True)
lads

,LADNM,geometry
0,Bolton,"POLYGON ((-2.51546 53.53532, -2.51524 53.535, ..."
1,Bury,"POLYGON ((-2.34446 53.54573, -2.34561 53.54639..."
2,Manchester,"POLYGON ((-2.25857 53.35787, -2.25845 53.35734..."
3,Oldham,"POLYGON ((-2.15654 53.49421, -2.15737 53.4944,..."
4,Rochdale,"POLYGON ((-2.19408 53.53164, -2.19694 53.53181..."
5,Salford,"POLYGON ((-2.4347 53.42193, -2.44593 53.41743,..."
6,Stockport,"POLYGON ((-2.19113 53.35339, -2.19148 53.3536,..."
7,Tameside,"POLYGON ((-2.11333 53.43907, -2.1146 53.43987,..."
8,Trafford,"POLYGON ((-2.34655 53.36755, -2.34776 53.36783..."
9,Wigan,"POLYGON ((-2.61773 53.48178, -2.61976 53.48177..."


In [3]:
# GMAL JSON tuple layout: [lon, lat, overall, level, bus, rail, metro, locallink]
GMAL_COLS = ['lon', 'lat', 'overall', 'level', 'bus', 'rail', 'metro', 'locallink']
SCORE_COLS = ['overall', 'bus', 'rail', 'metro', 'locallink']

def load_gmal(path: Path) -> gpd.GeoDataFrame:
    with open(path) as f:
        payload = json.load(f)
    df = pd.DataFrame(payload['data'], columns=GMAL_COLS)
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['lon'], df['lat']),
        crs=4326,
    )
    return gdf

gmal_files = {'2016': 'gmal_2016.json', '2026': 'gmal_2026.json'}
gmal_gdfs = {year: load_gmal(DATA_DIR / fname) for year, fname in gmal_files.items()}
{year: len(gdf) for year, gdf in gmal_gdfs.items()}

{'2016': 129109, '2026': 129109}

In [4]:
# Spatial join (point-in-polygon) and compute per-LAD means for each year
rows = []
for year, gdf in gmal_gdfs.items():
    joined = gpd.sjoin(gdf, lads, how='inner', predicate='within')
    grouped = (
        joined.groupby('LADNM')[SCORE_COLS]
        .mean()
        .round(3)
        .reset_index()
    )
    grouped['cell_count'] = joined.groupby('LADNM').size().values
    grouped['year'] = year
    rows.append(grouped)

result = pd.concat(rows, ignore_index=True)
result = result[['year', 'LADNM', 'cell_count', *SCORE_COLS]].sort_values(['year', 'LADNM']).reset_index(drop=True)
result

,year,LADNM,cell_count,overall,bus,rail,metro,locallink
0,2016,Bolton,13975,5.502,4.118,0.374,0.000,1.010
1,2016,Bury,9955,5.385,4.186,0.000,0.473,0.726
2,2016,Manchester,11567,16.423,11.884,1.202,1.696,1.642
3,2016,Oldham,14246,4.296,3.262,0.046,0.433,0.554
4,2016,Rochdale,15794,3.789,2.641,0.124,0.163,0.861
5,2016,Salford,9716,7.355,6.018,0.435,0.332,0.570
6,2016,Stockport,12602,5.920,4.814,0.745,0.023,0.338
7,2016,Tameside,10312,5.529,4.442,0.505,0.242,0.340
8,2016,Trafford,10600,6.335,4.881,0.233,0.830,0.391
9,2016,Wigan,18815,4.232,3.878,0.196,0.000,0.158


In [5]:
out_path = DATA_DIR / 'gmal_lad_means.csv'
result.to_csv(out_path, index=False)
print('Wrote', out_path, f'({out_path.stat().st_size} bytes)')

Wrote C:\Users\Lenovo\Desktop\UCL_Moodle\MT2\CASA0029-Urban_Data_Visualisation\Assignments\Group_Visualisation\CASA0029-Group-18-Bus-Delay-in-Manchester\data\accessibility\gmal_lad_means.csv (1033 bytes)
